# 06 - Relation-Specific Expert Routing (item 2)

Trains a 2-way router (dependency vs. non-dependency relations) instead of the originally proposed full 5-way mixture of experts -- see `relation_experts.py`'s module docstring for why: `supersedes` has only 83 examples corpus-wide, too few for its own dedicated learned expert without serious overfitting risk. The dependency/non-dependency split is the one the rule-based decision rule itself already branches on (Eq. 14), so it's the split most likely to carry a real difference in what evidence matters, and both sides have thousands of training examples.

Compares three things on the same held-out gold set as notebook 05: the rule, the single pooled learned model, and the routed 2-expert model -- including a per-route breakdown, since the interesting question is whether routing helps one relation category more than the other, not just the aggregate number.

In [ ]:
# == Environment + paths =========================================================
import json, os, re, sys, time
from pathlib import Path

# Kaggle detection uses an env var Kaggle actually sets, not path existence: on
# Windows, Path('/kaggle/working').exists() is NOT a reliable off-Kaggle check --
# pathlib treats a leading '/' as relative to the current drive's root, so an
# unrelated local C:\kaggle or F:\kaggle directory would silently return True.
ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.environ.get("KAGGLE_URL_BASE"))

def _find(name, prefer_largest=True):
    """Search for `name`; if multiple matches exist, prefer the largest file --
    a complete/authoritative cache is virtually always bigger than a stale or
    partial one left over from an earlier run."""
    roots = ["/kaggle/input"] if ON_KAGGLE else ["..", "../..", "."]
    hits = []
    for root in roots:
        r = Path(root)
        if r.exists():
            hits.extend(r.rglob(name))
    if not hits:
        return None
    if prefer_largest:
        hits.sort(key=lambda p: p.stat().st_size, reverse=True)
    return hits[0]

def _find_src_dir():
    """Locate the directory holding this project's shared modules (identified by
    containing graph_utils.py, not by a fixed relative path) -- on Kaggle, an
    attached dataset lands under /kaggle/input/<dataset-name>/..., so neither
    '../src' nor 'src' exists there; search by content the same way _find() above
    locates data files, instead of assuming a fixed layout."""
    for c in (Path("../src"), Path("src"), Path("/kaggle/working/src")):
        if (c / "graph_utils.py").exists():
            return c
    if ON_KAGGLE:
        hits = list(Path("/kaggle/input").rglob("graph_utils.py"))
        if hits:
            return hits[0].parent
    return Path("src")  # last resort; will raise ModuleNotFoundError on import if wrong

SRC_DIR = _find_src_dir()
sys.path.insert(0, str(SRC_DIR))

BENCH = _find("scireqa_full_relational_v1.1_humanmerged.jsonl") or _find("scireqa_full_relational_v1.1.jsonl")
REFS_CACHE = _find("citecheck_refs_cache.json")
CITE_RESULTS = _find("citecheck_results.json")
BENCH_VERIFIED = _find("scireqa_human_verified_subset.jsonl")
assert BENCH and REFS_CACHE and CITE_RESULTS, "attach the SciRecBench + CiteCheck output dataset"

OUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("../outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ON_KAGGLE:", ON_KAGGLE)
print("SRC_DIR:", SRC_DIR.resolve() if SRC_DIR.exists() else f"{SRC_DIR} (NOT FOUND -- attach the src/ dataset)")
print("BENCH:", BENCH, f"({BENCH.stat().st_size:,} bytes)" if BENCH else "")
print("REFS_CACHE:", REFS_CACHE, f"({REFS_CACHE.stat().st_size:,} bytes)" if REFS_CACHE else "")
print("CITE_RESULTS:", CITE_RESULTS)
print("BENCH_VERIFIED:", BENCH_VERIFIED)
print("OUT_DIR:", OUT_DIR)


In [ ]:
import json
import numpy as np
from graph_utils import CitationGraph, load_relational_claims, verify
from hard_negatives import build_tfidf_index
from verifier_features import build_dataset
from verifier_model import load_silver_labels, load_gold_labels, bootstrap_prf1, paired_bootstrap_diff
from relation_experts import route, split_by_route, train_routed_experts, predict_routed

graph = CitationGraph.load(REFS_CACHE)
all_claims = load_relational_claims(BENCH, CITE_RESULTS)

paper_abstracts = {}
with open(BENCH, encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        for gt in rec.get("ground_truth_papers") or []:
            pid, ab = gt.get("paper_id"), gt.get("abstract")
            if pid and ab:
                paper_abstracts[pid] = ab
tfidf_vectors = build_tfidf_index(paper_abstracts)

silver = load_silver_labels(CITE_RESULTS)
gold = load_gold_labels(BENCH_VERIFIED) if BENCH_VERIFIED else {}
silver_clean = {k: v for k, v in silver.items() if k not in gold}

X_train, y_train, meta_train = build_dataset(all_claims, graph, tfidf_vectors, silver_clean)
X_gold, y_gold, meta_gold = build_dataset(all_claims, graph, tfidf_vectors, gold)
print(f"Train: {X_train.shape} | Gold: {X_gold.shape}")

print("\nTraining routed experts...")
experts = train_routed_experts(X_train, y_train, meta_train)
routed_preds = predict_routed(experts, X_gold, meta_gold)

print("\nTraining single pooled model for comparison...")
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(X_train)
pooled = LogisticRegression(max_iter=2000, class_weight="balanced").fit(scaler.transform(X_train), y_train)
pooled_preds = pooled.predict(scaler.transform(X_gold))

rule_preds = np.array([1 if verify(m["paper_a"], m["relation"], m["paper_b"], graph) == "supported" else 0
                       for m in meta_gold])

results = {
    "rule_based": bootstrap_prf1(y_gold, rule_preds),
    "pooled_logreg": bootstrap_prf1(y_gold, pooled_preds),
    "routed_2expert_logreg": bootstrap_prf1(y_gold, routed_preds),
}
for name, r in results.items():
    print(f"{name}: precision={r['precision']:.3f} recall={r['recall']:.3f} f1={r['f1']:.3f} "
          f"(F1 CI [{r['f1_ci'][0]:.3f}, {r['f1_ci'][1]:.3f}])")

# Paired comparison, overall: same gold claims scored under both, more sensitive
# than eyeballing the two CIs above (see paired_bootstrap_diff's docstring).
overall_cmp = paired_bootstrap_diff(y_gold, pooled_preds, routed_preds)
results["paired_pooled_vs_routed"] = overall_cmp
sig = " *SIGNIFICANT*" if overall_cmp["f1_significant"] else ""
print(f"\nPaired pooled-vs-routed: F1 diff={overall_cmp['f1_diff']:+.3f} "
      f"CI=[{overall_cmp['f1_diff_ci'][0]:+.3f}, {overall_cmp['f1_diff_ci'][1]:+.3f}]{sig}")

for route_name in ("dependency", "non_dependency"):
    idx = [i for i, m in enumerate(meta_gold) if route(m["relation"]) == route_name]
    if not idx:
        continue
    yt = y_gold[idx]
    r_pooled = bootstrap_prf1(yt, pooled_preds[idx], n_boot=500)
    r_routed = bootstrap_prf1(yt, routed_preds[idx], n_boot=500)
    route_cmp = paired_bootstrap_diff(yt, pooled_preds[idx], routed_preds[idx], n_boot=500)
    sig = " *SIGNIFICANT*" if route_cmp["f1_significant"] else ""
    print(f"\n[{route_name}, n={len(idx)}] pooled F1={r_pooled['f1']:.3f} | routed F1={r_routed['f1']:.3f} "
          f"| paired diff={route_cmp['f1_diff']:+.3f} "
          f"CI=[{route_cmp['f1_diff_ci'][0]:+.3f}, {route_cmp['f1_diff_ci'][1]:+.3f}]{sig}")
    results[f"{route_name}_pooled"] = r_pooled
    results[f"{route_name}_routed"] = r_routed
    results[f"paired_{route_name}_pooled_vs_routed"] = route_cmp

with open(OUT_DIR / "relation_experts_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print("\nSaved to", OUT_DIR / "relation_experts_results.json")
